# Visual-LoRA and model ablations

Compare 24 configurations on held-out participant 3, using 15 coarse and 15 fine epochs. Fine-only adapters reuse a shared coarse checkpoint. Configurations that adapt the coarse stage or change the main architecture train from epoch one. The final cell runs the selected experiments.


In [ ]:
from dataclasses import dataclass, replace
from pathlib import Path
import json
import random
import torch

import things_eeg2.model as samga_model
import things_eeg2.train as trainer
from things_eeg2.visual_lora import VisualLoRAConfig

print("Python/PyTorch:", torch.__version__, "compiled CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

from things_eeg2.config import resolve_paths
from things_eeg2.paths import workspace_path
from things_eeg2.experiments import resume_config_differences


## Reference settings

The saved reference configuration supplies architecture and dataset settings. Set `REFERENCE_COARSE_CHECKPOINT` to a run whose `config.json` is available. The shared coarse weights used by fine-only runs are created separately with the 15-epoch schedule.


In [ ]:
REFERENCE_COARSE_CHECKPOINT = workspace_path("checkpoints/things_eeg2_samga/samga_9_to_1_20260924_003144/last.pt")
BASE_SETTINGS_FILE = REFERENCE_COARSE_CHECKPOINT.parent / "config.json"
EXPERIMENT_ROOT = workspace_path("checkpoints/things_eeg2_lora_sweep_15_15")
BASE_COARSE_CHECKPOINT: Path | None = None  
VIT_CHECKPOINT = workspace_path("models/ViT-B-16-laion2b/open_clip_model.safetensors")

if not BASE_SETTINGS_FILE.is_file() or not VIT_CHECKPOINT.is_file():
    raise FileNotFoundError("The reference configuration or pretrained ViT is missing.")
saved = json.loads(BASE_SETTINGS_FILE.read_text(encoding="utf-8"))
data_raw = dict(saved["data"])
data_raw["dataset_root"] = workspace_path(data_raw["dataset_root"])
data_raw["visual_feature_dir"] = workspace_path(data_raw["visual_feature_dir"])
visual_raw = dict(saved["model"]["visual"])
visual_raw["layer_ids"] = tuple(visual_raw["layer_ids"])
optimization_raw = dict(saved["optimization"])
optimization_raw["mmd_kernel_scales"] = tuple(optimization_raw["mmd_kernel_scales"])

BASE_CONFIG = trainer.RunConfig(
    data=trainer.DataConfig(**data_raw),
    model=samga_model.ModelConfig(
        eeg=samga_model.EEGEncoderConfig(**saved["model"]["eeg"]),
        visual=samga_model.VisualRouterConfig(**visual_raw),
    ),
    optimization=replace(trainer.OptimizationConfig(**optimization_raw),
                         coarse_epochs=15, epochs=30),
    checkpoint_root=EXPERIMENT_ROOT,
    run_name="subject03_lora_sweep",
)
status = trainer.visual_feature_status(BASE_CONFIG.data, BASE_CONFIG.model)
if not status["ready"]:
    raise RuntimeError(f"Selected cached ViT features are incomplete: {status}")
print("Fixed base: subject 3 held out; 100 images × 9 subjects; 15 coarse + 15 fine epochs.")
print("Feature layers:", BASE_CONFIG.model.visual.layer_ids)
print("Loss:", BASE_CONFIG.optimization.temperature, "temperature; MMD schedule",
      BASE_CONFIG.optimization.mmd_weight_start, "to",
      BASE_CONFIG.optimization.mmd_weight_end)


## Ablation definitions

Select experiment names in `RUN_IDS`. The LoRA group varies visual adaptation, while the main-model group holds a rank-32 Q/V adapter on blocks 3-5 fixed and changes one model setting. Gradient caching splits visual forwards/backwards into image chunks while retaining the full retrieval/MMD batch.


In [ ]:
@dataclass(frozen=True)
class Experiment:
    name: str
    stage: str  
    blocks: tuple[int, ...] = ()
    rank: int = 0
    cap: float | None = 0.05
    scratch: bool = False
    target_layers: tuple[int, ...] | None = None
    router_subject_dropout: float | None = None
    router_prior_strength: float | None = None
    eeg_temporal_filters: int | None = None
    eeg_temporal_kernel: int | None = None
    eeg_pooling_stride: int | None = None

MAIN_LORA_BLOCKS = (3, 4, 5)  
MAIN_LORA_RANK = 32

EXPERIMENTS = (
    
    Experiment("cached_baseline", "cached"),
    Experiment("fine_tail_r8", "fine", (7, 8, 9), 8),
    Experiment("fine_tail_r32", "fine", (7, 8, 9), 32),
    Experiment("fine_tail_r128", "fine", (7, 8, 9), 128),
    Experiment("fine_early_r32", "fine", (3, 4, 5, 6), 32),
    Experiment("fine_sparse_r32", "fine", (3, 5, 7, 9), 32),
    Experiment("fine_full_r32", "fine", (3, 4, 5, 6, 7, 8, 9), 32),
    Experiment("fine_full_r128", "fine", (3, 4, 5, 6, 7, 8, 9), 128),
    Experiment("fine_full_r32_cap2pct", "fine", (3, 4, 5, 6, 7, 8, 9), 32, 0.02),
    Experiment("fine_full_r32_uncapped", "fine", (3, 4, 5, 6, 7, 8, 9), 32, None),
    Experiment("both_tail_r32", "both", (7, 8, 9), 32),
    Experiment("both_full_r32", "both", (3, 4, 5, 6, 7, 8, 9), 32),
    
    Experiment("main_scratch_baseline", "fine", MAIN_LORA_BLOCKS, MAIN_LORA_RANK, scratch=True),
    Experiment("main_target_shallow_3_5", "fine", MAIN_LORA_BLOCKS, MAIN_LORA_RANK, scratch=True, target_layers=(3, 5)),
    Experiment("main_target_middle_5_7", "fine", MAIN_LORA_BLOCKS, MAIN_LORA_RANK, scratch=True, target_layers=(5, 7)),
    Experiment("main_target_deep_7_9", "fine", MAIN_LORA_BLOCKS, MAIN_LORA_RANK, scratch=True, target_layers=(7, 9)),
    Experiment("main_target_single_5", "fine", MAIN_LORA_BLOCKS, MAIN_LORA_RANK, scratch=True, target_layers=(5,)),
    Experiment("main_target_single_9", "fine", MAIN_LORA_BLOCKS, MAIN_LORA_RANK, scratch=True, target_layers=(9,)),
    Experiment("main_target_dense_2_to_10", "fine", MAIN_LORA_BLOCKS, MAIN_LORA_RANK, scratch=True,
               target_layers=tuple(range(2, 11))),
    Experiment("main_router_global_only", "fine", MAIN_LORA_BLOCKS, MAIN_LORA_RANK, scratch=True,
               router_subject_dropout=1.0),
    Experiment("main_router_flat_prior", "fine", MAIN_LORA_BLOCKS, MAIN_LORA_RANK, scratch=True,
               router_prior_strength=0.0),
    Experiment("main_eeg_filters_20", "fine", MAIN_LORA_BLOCKS, MAIN_LORA_RANK, scratch=True,
               eeg_temporal_filters=20),
    Experiment("main_eeg_kernel_11", "fine", MAIN_LORA_BLOCKS, MAIN_LORA_RANK, scratch=True,
               eeg_temporal_kernel=11),
    Experiment("main_eeg_pool_stride_3", "fine", MAIN_LORA_BLOCKS, MAIN_LORA_RANK, scratch=True,
               eeg_pooling_stride=3),
)
BY_NAME = {experiment.name: experiment for experiment in EXPERIMENTS}

RUN_IDS = list(BY_NAME)  
SWEEP_ORDER_SEED = 20260924
random.Random(SWEEP_ORDER_SEED).shuffle(RUN_IDS)
RUN_IDS = tuple(RUN_IDS)

VIT_MICROBATCH_IMAGES = 8  
VIT_EVAL_IMAGES = 8
LORA_LEARNING_RATE = 1e-4

def build_config(experiment: Experiment) -> trainer.RunConfig:
    eeg = BASE_CONFIG.model.eeg
    visual = BASE_CONFIG.model.visual
    if experiment.target_layers is not None:
        visual = replace(visual, layer_ids=experiment.target_layers,
                         prior_center=max(experiment.target_layers))
    if experiment.router_subject_dropout is not None:
        visual = replace(visual, subject_dropout=experiment.router_subject_dropout)
    if experiment.router_prior_strength is not None:
        visual = replace(visual, prior_strength=experiment.router_prior_strength)
    if experiment.eeg_temporal_filters is not None:
        eeg = replace(eeg, temporal_filters=experiment.eeg_temporal_filters)
    if experiment.eeg_temporal_kernel is not None:
        eeg = replace(eeg, temporal_kernel=experiment.eeg_temporal_kernel)
    if experiment.eeg_pooling_stride is not None:
        eeg = replace(eeg, pooling_stride=experiment.eeg_pooling_stride)
    model = replace(BASE_CONFIG.model, eeg=eeg, visual=visual)
    if experiment.stage == "cached":
        adapter = VisualLoRAConfig(enabled=False)
    else:
        adapter = VisualLoRAConfig(
            enabled=True,
            require_cuda=True,
            stage=experiment.stage,
            checkpoint=VIT_CHECKPOINT,
            blocks=experiment.blocks,
            rank=experiment.rank,
            alpha=2.0 * experiment.rank,
            max_update_ratio=experiment.cap,
            images_per_batch=BASE_CONFIG.data.images_per_batch,
            gradient_cache_microbatch_size=VIT_MICROBATCH_IMAGES,
            evaluation_image_batch_size=VIT_EVAL_IMAGES,
            learning_rate=LORA_LEARNING_RATE,
            gradient_checkpointing=True,
            pooling="cls",
        )
        adapter.selected_blocks(model.visual.layer_ids)
    source = (BASE_COARSE_CHECKPOINT if not experiment.scratch
              and experiment.stage in {"cached", "fine"} else None)
    config = replace(BASE_CONFIG, model=model, visual_lora=adapter,
                     run_name="subject03_" + experiment.name,
                     resume_checkpoint=source)
    feature_status = trainer.visual_feature_status(config.data, config.model)
    if not feature_status["ready"]:
        raise RuntimeError(f"Cached visual features missing for {experiment.name}: "
                           f"{feature_status['missing']} {feature_status['incomplete']}")
    return config

for name in RUN_IDS:
    item = BY_NAME[name]
    config = build_config(item)
    epochs_to_run = (config.optimization.coarse_epochs if not item.scratch
                     and item.stage in {"cached", "fine"}
                     else config.optimization.epochs)
    print(f"{name:30s} stage={item.stage:6s} rank={item.rank:3d} "
          f"blocks={str(item.blocks):24s} layers={str(config.model.visual.layer_ids):30s} "
          f"epochs={epochs_to_run}")


## Run selected experiments

The run order is shuffled reproducibly. Completed matching runs are skipped; interrupted runs resume from saved weights. This cell can take substantial GPU time.


In [ ]:
def prior_progress(config: trainer.RunConfig):
    config = resolve_paths(config)
    matches = []
    for folder in config.checkpoint_root.glob(config.run_name + "_*"):
        saved_config_file = folder / "config.json"
        saved_history_file = folder / "history.json"
        last_file = folder / "last.pt"
        if not all(path.is_file() for path in (saved_config_file, saved_history_file, last_file)):
            continue
        try:
            previous = json.loads(saved_config_file.read_text(encoding="utf-8"))
            history = json.loads(saved_history_file.read_text(encoding="utf-8"))
        except (OSError, ValueError):
            continue
        if not history:
            continue
        
        
        if resume_config_differences(previous, config):
            continue
        last_epoch = int(history[-1]["epoch"])
        if (config.visual_lora.enabled
                and last_epoch > config.optimization.coarse_epochs):
            
            
            try:
                payload = torch.load(last_file, map_location="cpu", weights_only=False,
                                     mmap=True)
            except (OSError, RuntimeError, EOFError) as exc:
                print(f"Ignoring unreadable checkpoint {last_file}: {exc}", flush=True)
                continue
            if not payload.get("visual_lora_state_dict"):
                print(f"Ignoring invalid LoRA run (no adapter weights): {folder}", flush=True)
                continue
            del payload
        matches.append((last_epoch, last_file.stat().st_mtime, folder))
    return max(matches, default=None)

def ensure_shared_coarse_checkpoint() -> Path:
    
    shared = replace(BASE_CONFIG,
                     optimization=replace(BASE_CONFIG.optimization, epochs=15),
                     visual_lora=VisualLoRAConfig(enabled=False),
                     run_name="subject03_shared_coarse15",
                     resume_checkpoint=None)
    progress = prior_progress(shared)
    if progress is not None:
        last_epoch, _, folder = progress
        if last_epoch >= 15:
            print(f"Using shared epoch-15 coarse checkpoint: {folder / 'last.pt'}", flush=True)
            return folder / "last.pt"
        shared = replace(shared, resume_checkpoint=folder / "last.pt")
        print(f"Resuming shared coarse model after epoch {last_epoch}", flush=True)
    else:
        print("Training shared coarse model for 15 epochs", flush=True)
    output = trainer.run_training(shared)
    return output / "last.pt"

EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)
if any(not BY_NAME[name].scratch and BY_NAME[name].stage in {"cached", "fine"}
       for name in RUN_IDS):
    BASE_COARSE_CHECKPOINT = ensure_shared_coarse_checkpoint()
summary_path = EXPERIMENT_ROOT / "sweep_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8")) if summary_path.exists() else {}

for name in RUN_IDS:
    item = BY_NAME[name]
    config = build_config(item)
    progress = prior_progress(config)
    if progress is not None:
        last_epoch, _, folder = progress
        if last_epoch >= config.optimization.epochs:
            print(f"Skipping complete {name}: {folder}")
            continue
        config = replace(config, resume_checkpoint=folder / "last.pt")
        print(f"Resuming {name} after epoch {last_epoch}: {config.resume_checkpoint}", flush=True)
    else:
        print(f"Starting {name} from " +
              ("shared epoch-15 coarse checkpoint" if config.resume_checkpoint else "epoch 1"),
              flush=True)
    output = trainer.run_training(config)
    history = json.loads((output / "history.json").read_text(encoding="utf-8"))
    fine = [row for row in history if row["epoch"] > config.optimization.coarse_epochs]
    last = history[-1]
    summary[name] = {
        "run_directory": str(output.resolve()),
        "last_epoch": last["epoch"],
        "final_r1": last["validation_test_r1"],
        "final_r5": last["validation_test_r5"],
        "best_fine_r1": max(row["validation_test_r1"] for row in fine),
        "best_fine_r5": max(row["validation_test_r5"] for row in fine),
    }
    summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print(f"Completed {name}: R@1={last['validation_test_r1']:.4f}, "
          f"R@5={last['validation_test_r5']:.4f}", flush=True)
print("Sweep summary:", summary_path.resolve())
